In [41]:
%load_ext autoreload

In [42]:
%autoreload 2

In [43]:
from rect_prune_ood.lib.eval.metrics import compute_traditional_ood
from rect_prune_ood.lib.experiments.run_experiment import (
	run_experiment_for_method,
	run_single_experiment,
    _run_imagenet_c_experiment,
)

In [44]:
from rect_prune_ood.lib.experiments.backbones import get_mobilenetV2, get_resnet50
from rect_prune_ood.lib.experiments.feature_extractor import FeatureExtractor
from rect_prune_ood.lib.experiments.mobilenet_feature_extractor import (
	MobileNetFeatureExtractor,
)
from rect_prune_ood.lib.experiments.pipeline import Pipeline
from rect_prune_ood.lib.util.data_loader import (
	get_data_loader,
	get_imagenet_c_subset_loader,
	id_datasets,
	ood_datasets,
	perturbed_datasets,
)
from rect_prune_ood.lib.util.paths import get_method_result_path

In [54]:
method_name = "scale"
base_dataset = "imagenet"
backbone = "resnet50"

result_dir = get_method_result_path(base_dataset, backbone, method_name)
print(result_dir)

imagenet_c_pertubations = [
		'brightness',
		'defocus_blur',
		'fog',
		'gaussian_blur',
		'glass_blur',
		'jpeg_compression',
		'pixelate',
		'shot_noise',
		'spatter',
		'zoom_blur',
		'contrast',
		'elastic_transform',
		'frost',
		'gaussian_noise',
		'impulse_noise',
		'motion_blur',
		'saturate',
		'snow',
		'speckle_noise',
	]

/visinf/home/vilab18/RectPruneOOD/results/imagenet/resnet50/scale


In [ ]:
run_experiment_for_method(
	backbone='resnet50',
	layer='avgpool',
	method_name='ashp@90',
	# capture_activations=True,
)

In [55]:
import numpy as np
all_perturbed_scores = []
all_perturbed_labels = []

#get all perturbed scores and labels
for perturbation in imagenet_c_pertubations:
    #print(np.load(result_dir + "/scores/imagenet-c_sev1_" + perturbation + ".npy", allow_pickle=True).item()["energy_score"].shape)
    all_perturbed_scores.append(
        np.load(result_dir + "/scores/imagenet-c_sev1_" + perturbation + ".npy", allow_pickle=True).item()["energy_score"]
    )
    all_perturbed_labels.append(
        np.load(result_dir + "/scores/imagenet-c_sev1_" + perturbation + ".npy", allow_pickle=True).item()["label"]
    )

#also get normal ID scores
all_perturbed_scores.append(np.load(
    result_dir + f"/scores/{base_dataset}.npy", allow_pickle=True).item()["energy_score"]
)
all_perturbed_labels.append(np.load(
    result_dir + f"/scores/{base_dataset}.npy", allow_pickle=True).item()["label"]
)

id_scores = np.concatenate(all_perturbed_scores, axis=0)
id_labels = np.concatenate(all_perturbed_labels, axis=0)


In [56]:
from rect_prune_ood.lib.eval.metrics import cal_metric, print_all_results


ood_datasets = ["inat", "sun", "places", "dtd", "ninco", "ssb-hard"]
all_results = []


# calculate results for all datasets
for ood_dataset in ood_datasets:
	ood_scores = np.load(
		result_dir + f"/scores/{ood_dataset}.npy", allow_pickle=True
	).item()["energy_score"]
	

	results = cal_metric(id_scores, ood_scores, method_name)
	results['ood_dataset'] = ood_dataset
	all_results.append(results)


print(f' ID dataset: {base_dataset}')
print_all_results(all_results, ood_datasets, method_name)

 ID dataset: imagenet
 OOD detection method: scale
              FPR    AUROC  AUIN  
inat          40.97  92.03  99.91
sun           62.05  84.57  99.79
places        73.44  78.86  99.69
dtd           43.17  90.41  99.93
ninco         88.16  63.57  99.65
ssb-hard      94.65  53.24  96.09
AVG           67.07  77.11  99.18
